# Esta parte do código se refere à pipeline da camada GOLD em BATCH para testes antes de subir ao AWS

In [235]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Instalando as dependências
# ~~~~~~~~~~~~~~~~~~~~~~~~~~

# pyarrow para salvar em PARQUET

!pip install pyarrow colorama tabulate --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\carol\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [236]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Importações
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
import logging
import time
import os
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

In [237]:
# ~~~~~~~~~~~~~~~
# CONFIGURAÇÕES
# ~~~~~~~~~~~~~~~
from pathlib import Path
from datetime import datetime

DATA_SILVER = Path("silver")
DATA_GOLD = Path("gold")

DATA_GOLD.mkdir(parents=True, exist_ok=True)

PROCESSAMENTO = datetime.now()

In [238]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONFIGURAÇÃO DOS LOGS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s"
)

log = logging.getLogger(__name__)

In [239]:
# ~~~~~~~~~~~~~~~
# LOG INICIAL
# ~~~~~~~~~~~~~~~

log.info("~" * 35)
log.info("INICIANDO ETL DA CAMADA GOLD")
log.info("~" * 35)

2026-07-12 23:37:41,398 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-07-12 23:37:41,399 | INFO     | INICIANDO ETL DA CAMADA GOLD
2026-07-12 23:37:41,400 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# OBSERVABILIDADE: MÉTRICAS ESTRUTURADAS E ALERTAS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# (mesmo padrão usado nas camadas Bronze e Silver)

def log_metrica(evento, **campos):
    """
    Loga um evento estruturado (campo=valor), permitindo consulta via
    CloudWatch Logs Insights, ex.:
        fields @timestamp, tabela, volume, latencia_segundos
        | filter evento = "tabela_processada"
    """
    campos_formatados = " | ".join(f"{chave}={valor}" for chave, valor in campos.items())
    log.info(f"[METRICA] evento={evento} | {campos_formatados}")


def emitir_alerta(mensagem, **contexto):
    """
    Emite um alerta de erro (sempre em nível ERROR no log) e tenta
    publicar em um tópico SNS, se configurado via variável de ambiente
    SNS_TOPIC_ARN, para que a falha não dependa de alguém checar o log
    manualmente.
    """
    contexto_formatado = " | ".join(f"{k}={v}" for k, v in contexto.items())
    log.error(f"[ALERTA] {mensagem} | {contexto_formatado}")

    topico_sns = os.environ.get("SNS_TOPIC_ARN")

    if not topico_sns:
        log.warning("[ALERTA] SNS_TOPIC_ARN não configurado - alerta ficou registrado apenas no log")
        return

    try:
        import boto3
        sns = boto3.client("sns")
        sns.publish(
            TopicArn=topico_sns,
            Subject="[Tech Challenge] Falha no pipeline",
            Message=f"{mensagem}\n\n{contexto_formatado}"
        )
        log.info("[ALERTA] Notificação SNS publicada com sucesso")
    except Exception as e:
        log.warning(f"[ALERTA] Falha ao publicar no SNS (alerta permanece apenas no log): {e}")

In [240]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# LENDO ARQUIVOS DA CAMADA SILVER
"""
    Lê um arquivo Parquet da camada SILVER.

    Args:
        tabela (str): Nome da tabela.
    """
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def ler_silver(tabela):

    caminho = DATA_SILVER / f"{tabela}.parquet"

    log.info(f"Lendo Silver: {caminho}")

    return pd.read_parquet(caminho)

In [241]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - RANKING UF
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_ranking_uf():

    log.info("Construindo Gold: Ranking UF")

    df = ler_silver("uf")

    # removendo o Total, pois só há uma única informação
    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ].copy()

    # ordena
    df = df.sort_values(
        by=["ano", "rede", "taxa_alfabetizacao"],
        ascending=[True, True, False]
    )

    # ranking por ano e rede
    df["ranking"] = (
        df.groupby(
            ["ano", "rede"]
        )["taxa_alfabetizacao"]
        .rank(
            method="dense",
            ascending=False
        )
        .astype(int)
    )

    df["_gold_processed_at"] = datetime.now()
    
    df = df[
        [
            "ano",
            "sigla_uf",
            "sigla_uf_nome",
            "rede",
            "taxa_alfabetizacao",
            "ranking",
            "_gold_processed_at"
        ]
    ]

    log.info("Ranking UF criado")

    return df

In [242]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - RANKING MUNICÍPIOS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_ranking_municipio():

    log.info("Construindo Gold: Ranking Municípios")

    df = ler_silver("municipio")

    # Ordena por ano, rede e taxa
    df = df.sort_values(
        by=["ano", "rede", "taxa_alfabetizacao"],
        ascending=[True, True, False]
    )

    # Cria ranking por ano e rede
    df["ranking"] = (
        df.groupby(
            ["ano", "rede"]
        )["taxa_alfabetizacao"]
        .rank(
            method="dense",
            ascending=False
        )
        .astype(int)
    )

    df["_gold_processed_at"] = datetime.now()

    # Mantém somente as colunas importantes
    df = df[
        [
            "ano",
            "id_municipio",
            "id_municipio_nome",
            "rede",
            "taxa_alfabetizacao",
            "ranking",
            "_gold_processed_at"
        ]
    ]

    log.info("Ranking de Municípios criado")

    return df

In [243]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - EVOLUÇÃO UF
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def gold_evolucao_uf():

    log.info("Construindo Gold: Evolução UF")

    df = ler_silver("uf")

    df = df[
        [
            "ano",
            "sigla_uf",
            "sigla_uf_nome",
            "rede",
            "taxa_alfabetizacao",
            "media_portugues"
        ]
    ].copy()

    df["_gold_processed_at"] = datetime.now()

    df = df.sort_values(
        by=["sigla_uf", "rede", "ano"]
    )

    log.info("Gold Evolução UF criada")

    return df

In [251]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - RESUMO POR REDE
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_resumo_rede():

    log.info("Construindo Gold: Resumo por Rede")

    df = ler_silver("uf")
    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ]

    df_gold = (        
        df.groupby(["ano", "rede"])
          .agg(
              media_taxa_alfabetizacao=(
                  "taxa_alfabetizacao",
                  "mean"
              ),
              media_portugues=(
                  "media_portugues",
                  "mean"
              ),
              quantidade_ufs=(
                  "sigla_uf",
                  "nunique"
              )
          )
          .reset_index()
    )
    
    df_gold["media_taxa_alfabetizacao"] = (
    df_gold["media_taxa_alfabetizacao"].round(2)
    )

    df_gold["media_portugues"] = (
        df_gold["media_portugues"].round(2)
    )

    df_gold["_gold_processed_at"] = datetime.now()

    log.info("Resumo por Rede criado")

    return df_gold

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# HELPER: EXTRAI A META DO PRÓPRIO ANO DA LINHA
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# As tabelas de meta vêm em formato LARGO: uma coluna por ano-alvo
# (meta_alfabetizacao_2024 ... meta_alfabetizacao_2030) na mesma linha
# que carrega a taxa observada (`taxa_alfabetizacao`) daquele ano. Para
# comparar "resultado do ano X" com "meta do ano X", é preciso pegar,
# em cada linha, a coluna de meta cujo ano bate com o `ano` da própria
# linha.

def _extrair_meta_do_ano(df):
    """
    Args:
        df (pandas.DataFrame): Tabela de meta (uf ou município), contendo
            a coluna `ano` e as colunas `meta_alfabetizacao_2024..2030`.

    Returns:
        pandas.Series: Valor da meta definida para o próprio `ano` da
            linha (NaN se não houver meta definida para aquele ano, ex.:
            anos anteriores a 2024).
    """

    colunas_meta = {
        ano: f"meta_alfabetizacao_{ano}"
        for ano in range(2024, 2031)
        if f"meta_alfabetizacao_{ano}" in df.columns
    }

    meta_do_ano = pd.Series(np.nan, index=df.index, dtype="float64")

    for ano, coluna in colunas_meta.items():
        mascara = df["ano"] == ano
        meta_do_ano.loc[mascara] = df.loc[mascara, coluna]

    return meta_do_ano

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - COMPARAÇÃO META VS. RESULTADO (UF)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_comparacao_meta_uf():

    log.info("Construindo Gold: Comparação Meta vs. Resultado - UF")

    df = ler_silver("meta_alfabetizacao_uf")

    # removendo o Total, mesmo critério usado no ranking/evolução de UF
    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ].copy()

    df["meta_do_ano"] = _extrair_meta_do_ano(df)

    # só faz sentido comparar quando existe meta definida para aquele ano
    df = df[df["meta_do_ano"].notna()].copy()

    df["diferenca_pp"] = (
        df["taxa_alfabetizacao"] - df["meta_do_ano"]
    ).round(2)

    df["atingiu_meta"] = df["diferenca_pp"] >= 0

    df["_gold_processed_at"] = datetime.now()

    df = df[
        [
            "ano",
            "sigla_uf",
            "sigla_uf_nome",
            "rede",
            "taxa_alfabetizacao",
            "meta_do_ano",
            "diferenca_pp",
            "atingiu_meta",
            "_gold_processed_at"
        ]
    ]

    df = df.sort_values(
        by=["sigla_uf", "rede", "ano"]
    )

    log.info("Gold Comparação Meta vs. Resultado (UF) criada")

    return df

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - COMPARAÇÃO META VS. RESULTADO (MUNICÍPIO)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_comparacao_meta_municipio():

    log.info("Construindo Gold: Comparação Meta vs. Resultado - Município")

    df = ler_silver("meta_alfabetizacao_municipio")

    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ].copy()

    df["meta_do_ano"] = _extrair_meta_do_ano(df)

    df = df[df["meta_do_ano"].notna()].copy()

    df["diferenca_pp"] = (
        df["taxa_alfabetizacao"] - df["meta_do_ano"]
    ).round(2)

    df["atingiu_meta"] = df["diferenca_pp"] >= 0

    df["_gold_processed_at"] = datetime.now()

    df = df[
        [
            "ano",
            "id_municipio",
            "id_municipio_nome",
            "rede",
            "taxa_alfabetizacao",
            "meta_do_ano",
            "diferenca_pp",
            "atingiu_meta",
            "nivel_alfabetizacao",
            "_gold_processed_at"
        ]
    ]

    df = df.sort_values(
        by=["id_municipio", "rede", "ano"]
    )

    log.info("Gold Comparação Meta vs. Resultado (Município) criada")

    return df

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - BASE DE ALUNOS PARA MODELAGEM (ALFABETIZAÇÃO)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Esta é a base de referência para a Fase 3 (modelo supervisionado que
# prevê `alfabetizado`). Vem da integração alunos + município + UF feita
# na Silver (`alunos_integrado`), no nível de granularidade certo: um
# registro por aluno.

# AVISO DE DATA LEAKAGE: as colunas abaixo são mantidas nesta tabela por
# transparência/rastreabilidade, mas NÃO PODEM entrar como feature no
# treinamento do modelo -- cada uma vaza informação do próprio alvo.
COLUNAS_RISCO_DATA_LEAKAGE = {
    "proficiencia": (
        "É a variável usada para DEFINIR o alvo `alfabetizado` "
        "(corte de proficiência). Usá-la como feature vaza o alvo "
        "diretamente."
    ),
    "taxa_alfabetizacao_municipio": (
        "Agregado que já inclui o resultado do próprio aluno no cálculo "
        "(leakage indireto: o valor 'carrega' a resposta)."
    ),
    "taxa_alfabetizacao_uf": (
        "Mesmo motivo acima, em nível estadual."
    ),
}


def gold_alunos_alfabetizacao():

    log.info("Construindo Gold: Base de Alunos para Modelagem (Alfabetização)")

    df = ler_silver("alunos_integrado")

    # mantém só alunos com o alvo definido: a avaliação não é aplicada a
    # 100% dos alunos (ausentes, caderno não preenchido etc.) -- sem
    # `alfabetizado` preenchido, o registro não serve para treino
    # supervisionado
    antes = len(df)
    df = df[df["alfabetizado"].notna()].copy()
    removidos = antes - len(df)

    if removidos > 0:
        log.info(
            f"Removidos {removidos} registro(s) sem alfabetizado definido "
            f"(aluno não avaliado)"
        )

    df["_gold_processed_at"] = datetime.now()

    colunas_finais = [
        "ano",
        "id_aluno",
        "id_escola",
        "id_municipio",
        "id_municipio_nome",
        "sigla_uf",
        "sigla_uf_nome",
        "serie",
        "rede",
        "presenca",
        "preenchimento_caderno",
        "peso_aluno",
        "taxa_alfabetizacao_municipio",
        "media_portugues_municipio",
        "taxa_alfabetizacao_uf",
        "media_portugues_uf",
        "proficiencia",
        "alfabetizado",
        "_gold_processed_at",
    ]

    colunas_finais = [c for c in colunas_finais if c in df.columns]

    df = df[colunas_finais]

    log.warning(
        f"[DATA LEAKAGE] Esta tabela contém colunas que NÃO podem ser "
        f"usadas como feature: {list(COLUNAS_RISCO_DATA_LEAKAGE.keys())}. "
        f"Ver COLUNAS_RISCO_DATA_LEAKAGE para o motivo de cada uma."
    )

    log.info(
        f"Gold Base de Alunos para Modelagem criada: {len(df)} registro(s)"
    )

    return df

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# REGRAS DE DATA QUALITY
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
CHECKS = {

    "gold_ranking_uf": [

        {
            "tipo": "min_count",
            "valor": 100,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "sigla_uf",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "ranking",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0,100),
            "critico": True
        }

    ],
    "ranking_municipio": [

        {
            "tipo": "min_count",
            "valor": 1000,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "id_municipio",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "ranking",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        }
    ],
    "evolucao_uf": [

        {
            "tipo": "min_count",
            "valor": 100,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "sigla_uf",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "ano",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        }

    ],
    "resumo_rede": [

    {
        "tipo": "min_count",
        "valor": 7,
        "critico": True
    },

    {
        "tipo": "not_null",
        "coluna": "ano",
        "critico": True
    },

    {
        "tipo": "not_null",
        "coluna": "rede",
        "critico": True
    },

    {
        "tipo": "range",
        "coluna": "media_taxa_alfabetizacao",
        "valor": (0, 100),
        "critico": True
    }

],

    "comparacao_meta_uf": [

        {
            # valor conservador: como só existem metas para 2024-2030,
            # o volume real depende de quantos desses anos já têm
            # avaliação (ano) registrada na base. Ajustar após a
            # primeira execução com dados reais.
            "tipo": "min_count",
            "valor": 10,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "sigla_uf",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "meta_do_ano",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "meta_do_ano",
            "valor": (0, 100),
            "critico": True
        }

    ],

    "comparacao_meta_municipio": [

        {
            "tipo": "min_count",
            "valor": 50,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "id_municipio",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "meta_do_ano",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "meta_do_ano",
            "valor": (0, 100),
            "critico": True
        }

    ],

    "alunos_alfabetizacao": [

        {
            "tipo": "min_count",
            "valor": 100,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "id_aluno",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "alfabetizado",
            "critico": True
        },

        {
            "tipo": "unique",
            "coluna": "id_aluno",
            "critico": False
        }

    ]

}

In [246]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO DE QUALIDADE DA CAMADA GOLD
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def checar_qualidade(df, checks):

    log.info("Iniciando verificações de qualidade")

    for check in checks:

        if check["tipo"] == "min_count":

            assert len(df) >= check["valor"], \
                f"Quantidade mínima não atendida ({len(df)} registros)."

        elif check["tipo"] == "not_null":

            coluna = check["coluna"]

            assert df[coluna].isnull().sum() == 0, \
                f"Existem valores nulos na coluna '{coluna}'."

        elif check["tipo"] == "range":

            coluna = check["coluna"]
            minimo, maximo = check["valor"]

            assert (
                df[coluna].between(minimo, maximo).all()
            ), f"Valores fora do intervalo na coluna '{coluna}'."

    log.info("Checks de qualidade concluídos com sucesso!")

In [247]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# SALVA UM DATAFRAME NA CAMADA GOLD EM FORMATO PARQUET
"""
    Args:
        df (pandas.DataFrame): DataFrame tratado.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def salvar_gold(df, nome):

    caminho = DATA_GOLD / f"{nome}.parquet"

    df.to_parquet(
        caminho,
        index=False
    )

    log.info(f"Camada GOLD salva em {caminho}")

    return caminho

In [248]:
# ~~~~~~~~~~~~~~~~~~~~~~~~
# EXECUÇÃO DA CAMADA GOLD
"""
    Cada dataset Gold é construído de forma isolada: se um deles falhar,
    um alerta é emitido e os demais continuam sendo processados.
    Latência e volume de cada dataset são registrados como métricas
    estruturadas e consultáveis.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~
def executar_gold():

    log.info("~" * 35)
    log.info("INICIANDO CAMADA GOLD")
    log.info("~" * 35)

    inicio_pipeline = time.perf_counter()

    # Cada item: (nome do dataset salvo, função que constrói o dataframe, chave em CHECKS)
    datasets = [
        ("ranking_uf", gold_ranking_uf, "gold_ranking_uf"),
        ("ranking_municipio", gold_ranking_municipio, "ranking_municipio"),
        ("evolucao_uf", gold_evolucao_uf, "evolucao_uf"),
        ("resumo_rede", gold_resumo_rede, "resumo_rede"),
        ("comparacao_meta_uf", gold_comparacao_meta_uf, "comparacao_meta_uf"),
        ("comparacao_meta_municipio", gold_comparacao_meta_municipio, "comparacao_meta_municipio"),
        ("alunos_alfabetizacao", gold_alunos_alfabetizacao, "alunos_alfabetizacao"),
    ]

    datasets_ok = 0
    datasets_falha = 0

    for nome, funcao_construtora, chave_checks in datasets:

        log.info(f"Checando qualidade de {nome}")

        inicio_dataset = time.perf_counter()

        try:

            df_gold = funcao_construtora()

            checar_qualidade(
                df_gold,
                CHECKS[chave_checks]
            )

            salvar_gold(
                df_gold,
                nome
            )

            latencia_segundos = round(time.perf_counter() - inicio_dataset, 2)

            log_metrica(
                "tabela_processada",
                camada="gold",
                tabela=nome,
                volume=len(df_gold),
                latencia_segundos=latencia_segundos,
                status="sucesso"
            )

            datasets_ok += 1

        except Exception as e:

            latencia_segundos = round(time.perf_counter() - inicio_dataset, 2)

            log_metrica(
                "tabela_processada",
                camada="gold",
                tabela=nome,
                volume=0,
                latencia_segundos=latencia_segundos,
                status="falha"
            )

            emitir_alerta(
                f"Falha na construção do dataset Gold '{nome}'",
                camada="gold",
                tabela=nome,
                erro=str(e)
            )

            datasets_falha += 1

            # isola a falha: segue para o próximo dataset
            continue

    latencia_total_segundos = round(time.perf_counter() - inicio_pipeline, 2)

    log_metrica(
        "pipeline_concluido",
        camada="gold",
        tabelas_ok=datasets_ok,
        tabelas_falha=datasets_falha,
        latencia_total_segundos=latencia_total_segundos
    )

    if datasets_falha > 0:
        emitir_alerta(
            f"Pipeline Gold concluído com {datasets_falha} falha(s) de {datasets_ok + datasets_falha} dataset(s)",
            camada="gold",
            datasets_falha=datasets_falha,
            datasets_ok=datasets_ok
        )

    log.info("Camada Gold concluída!")

In [249]:
executar_gold()

2026-07-12 23:37:41,521 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-07-12 23:37:41,521 | INFO     | INICIANDO CAMADA GOLD
2026-07-12 23:37:41,522 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-07-12 23:37:41,523 | INFO     | Construindo Gold: Ranking UF
2026-07-12 23:37:41,523 | INFO     | Lendo Silver: silver\uf.parquet
2026-07-12 23:37:41,533 | INFO     | Ranking UF criado
2026-07-12 23:37:41,534 | INFO     | Checando qualidade de Ranking UF
2026-07-12 23:37:41,534 | INFO     | Iniciando verificações de qualidade
2026-07-12 23:37:41,536 | INFO     | Checks de qualidade concluídos com sucesso!
2026-07-12 23:37:41,541 | INFO     | Camada GOLD salva em gold\ranking_uf.parquet
2026-07-12 23:37:41,542 | INFO     | Checando qualidade de Ranking Municipio
2026-07-12 23:37:41,543 | INFO     | Construindo Gold: Ranking Municípios
2026-07-12 23:37:41,544 | INFO     | Lendo Silver: silver\municipio.parquet
2026-07-12 23:37:41,575 | INFO     | Ranking de Municípios criado
2026-0